###Ingestion del archivo "language_role.json"

####Paso 1 - Leer el archivo

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
languages_roles_schema = StructType(fields={
    StructField("roleId", IntegerType(), True),
    StructField("languageRole", StringType(), True)
})

In [0]:
languages_roles_df = spark.read \
                    .schema(languages_roles_schema) \
                    .option("multiLine", True) \
                    .json(f"abfss://bronze@moviehistory8.dfs.core.windows.net/{v_file_date}/language_role.json")

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
languages_roles_final_df = languages_roles_df \
                            .withColumnsRenamed({"roleId": "role_id",
                                                "languageRole": "language_role"}) \
                            .withColumn("ingestion_date", current_timestamp()) \
                            .withColumn("enviroment", lit("Production")) \
                            .withColumn("file_date", lit(v_file_date)) \
                            .withColumn("file_date", lit(v_file_date))

####Paso 3 - Escribir la salida en formato Parquet

In [0]:
#languages_roles_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistory8.dfs.core.windows.net/languages_roles")

In [0]:
languages_roles_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages_roles")

In [0]:
%sql
SELECT * FROM movie_silver.languages_roles;

language_role,role_id,ingestion_date,enviroment,file_date
Original,1,2026-09-26T16:18:38.457361Z,Production,2024-12-16
Spoken,2,2026-09-26T16:18:38.457361Z,Production,2024-12-16


In [0]:
#display(spark.read.parquet("abfss://silver@moviehistory8.dfs.core.windows.net/languages_roles"))

In [0]:
dbutils.notebook.exit("Success")